# Session 5 · First look at pandas

Work top to bottom and run each cell with **Shift+Enter**.

Cells under a **Your turn** heading contain `...` placeholders. Replace them, run the cell, then run the check cell beneath it. Checks print `PASS` or `NOT YET` and never stop the notebook.

Solutions are in the matching `_solutions` notebook. Open it only after a genuine attempt.

A pandas **DataFrame** is R's data frame or tibble. A single column is a **Series**, roughly a named vector. Unlike Python lists, Series and DataFrames **are vectorised**, so the R instincts you parked in Session 2 come back here.

In [85]:
def check(label, get, expected):
    """Print PASS or NOT YET. `get` is a lambda, so missing names or errors never stop the notebook."""
    try:
        got = get()
    except Exception as e:
        print(f"NOT YET  {label}: {type(e).__name__}: {e}")
        return
    try:
        ok = got.equals(expected) if hasattr(got, "equals") else bool(got == expected)
        if isinstance(expected, float) and isinstance(got, (int, float)):
            ok = abs(got - expected) < 1e-6
    except Exception:
        ok = False
    print(f"PASS     {label}" if ok else f"NOT YET  {label}: expected {expected!r}, got {got!r}")

## 1. Load and inspect

In [86]:
import pandas as pd

matches = pd.read_csv("../data/matches_2025_synthetic.csv", parse_dates=["date"])
matches.head()

,match_id,season,round,date,home_team,away_team,home_goals,home_behinds,away_goals,away_behinds,crowd
0,1,2025,1,2025-03-16,Collingwood,Carlton,12,12,19,11,14893.0
1,2,2025,1,2025-03-16,Hawthorn,Brisbane,14,9,17,10,12927.0
2,3,2025,1,2025-03-15,North Melbourne,Richmond,16,11,10,9,21940.0
3,4,2025,1,2025-03-15,Essendon,West Coast,10,12,13,4,64499.0
4,5,2025,1,2025-03-15,GWS,Fremantle,4,10,14,9,33167.0


In [87]:
print(matches.shape)           # dim()
print(list(matches.columns))   # names()
matches.dtypes                 # sapply(df, class)

(207, 11)
['match_id', 'season', 'round', 'date', 'home_team', 'away_team', 'home_goals', 'home_behinds', 'away_goals', 'away_behinds', 'crowd']


match_id                 int64
season                   int64
round                    int64
date            datetime64[us]
home_team                  str
away_team                  str
home_goals               int64
home_behinds             int64
away_goals               int64
away_behinds             int64
crowd                  float64
dtype: object

In [88]:
matches.tail(3)                # tail(df, 3)

,match_id,season,round,date,home_team,away_team,home_goals,home_behinds,away_goals,away_behinds,crowd
204,205,2025,23,2025-08-15,Carlton,North Melbourne,7,5,16,11,36933.0
205,206,2025,23,2025-08-16,Adelaide,Collingwood,24,8,11,11,47020.0
206,207,2025,23,2025-08-16,Melbourne,Fremantle,10,13,15,12,32094.0


## 2. Selecting columns

| dplyr / base R | pandas |
| --- | --- |
| `df$home_team` | `df["home_team"]` (a Series) |
| `select(df, home_team, away_team)` | `df[["home_team", "away_team"]]` (a DataFrame) |
| `select(df, -crowd)` | `df.drop(columns="crowd")` |

The double brackets matter: the outer pair means "select", the inner pair is a Python list of names.

In [89]:
matches["home_team"].head()

0        Collingwood
1           Hawthorn
2    North Melbourne
3           Essendon
4                GWS
Name: home_team, dtype: str

In [90]:
matches[["round", "home_team", "away_team"]].head()

,round,home_team,away_team
0,1,Collingwood,Carlton
1,1,Hawthorn,Brisbane
2,1,North Melbourne,Richmond
3,1,Essendon,West Coast
4,1,GWS,Fremantle


## 3. Filtering rows

A comparison on a column gives a Series of `True`/`False` values, a **boolean mask**. Putting the mask inside `df[...]` keeps the `True` rows, like `filter()`.

| dplyr | pandas |
| --- | --- |
| `filter(df, round == 1)` | `df[df["round"] == 1]` |
| `filter(df, round == 1, home_team == "Geelong")` | `df[(df["round"] == 1) & (df["home_team"] == "Geelong")]` |
| `filter(df, home_team %in% c("Geelong", "Sydney"))` | `df[df["home_team"].isin(["Geelong", "Sydney"])]` |
| any of the above | `df.query("round == 1 and home_team == 'Geelong'")` |

Use `&` and `|` (not `and`, `or`, `&&`), and wrap **each** condition in brackets.

In [91]:
round_5 = matches[matches["round"] == 5]
print(len(round_5))
round_5.head(3)

9


,match_id,season,round,date,home_team,away_team,home_goals,home_behinds,away_goals,away_behinds,crowd
36,37,2025,5,2025-04-11,Collingwood,St Kilda,12,8,12,6,41974.0
37,38,2025,5,2025-04-12,North Melbourne,Adelaide,15,11,9,11,31996.0
38,39,2025,5,2025-04-13,Gold Coast,Essendon,14,8,10,5,13009.0


In [92]:
matches.query("round == 5 and home_goals >= 15")

,match_id,season,round,date,home_team,away_team,home_goals,home_behinds,away_goals,away_behinds,crowd
37,38,2025,5,2025-04-12,North Melbourne,Adelaide,15,11,9,11,31996.0
39,40,2025,5,2025-04-10,Hawthorn,GWS,17,21,8,10,48352.0


### Your turn 3.1

- `round_1`: all round 1 matches
- `geelong_home`: all matches where Geelong was the home team
- `geelong_early`: Geelong home matches in rounds 1 to 10 (combine two conditions with `&`)

In [93]:
round_1 = matches[matches["round"] == 1]
geelong_home = matches[matches["home_team"] == "Geelong"]
geelong_early = geelong_home[geelong_home["round"] <= 10]

In [94]:
check("3.1 round_1", lambda: len(round_1), 9)
check("3.1 geelong_home", lambda: len(geelong_home), 11)
check("3.1 geelong_early", lambda: len(geelong_early), 3)

PASS     3.1 round_1
PASS     3.1 geelong_home
PASS     3.1 geelong_early


## 4. New columns

Column arithmetic is vectorised, exactly like `mutate()`. Assigning to a new name adds a column:

`df["new"] = df["a"] + df["b"]` is `df <- mutate(df, new = a + b)`.

In [95]:
matches["total_goals"] = matches["home_goals"] + matches["away_goals"]
matches[["home_team", "away_team", "total_goals"]].head()

,home_team,away_team,total_goals
0,Collingwood,Carlton,31
1,Hawthorn,Brisbane,31
2,North Melbourne,Richmond,26
3,Essendon,West Coast,23
4,GWS,Fremantle,18


### Your turn 4.1

Add three columns:

- `home_score`: home goals × 6 + home behinds
- `away_score`: the same for the away team
- `margin`: home score minus away score

Then set `mean_margin` to the mean margin rounded to 2 decimal places. A positive number means home teams won by that much on average.

In [96]:
matches["home_score"] = matches["home_goals"] * 6 + matches["home_behinds"]
matches["away_score"] = matches["away_goals"] * 6 + matches["away_behinds"]
matches["margin"] = matches["home_score"] - matches["away_score"]

mean_margin = matches["margin"].mean() # to 2 decimal places
mean_margin = round(mean_margin, 2)

In [97]:
check("4.1 mean_margin", lambda: mean_margin, 5.84)

PASS     4.1 mean_margin


### Your turn 4.2

Set `n_big_wins` to the number of matches decided by 60 points or more, either way. `.abs()` works on a Series, and `.sum()` on a True/False Series counts the `True` values, just like `sum()` on a logical vector in R.

In [98]:
n_big_wins = (matches["margin"].abs() >= 60).sum()

In [99]:
check("4.2 n_big_wins", lambda: n_big_wins, 16)

PASS     4.2 n_big_wins


## 5. Missing values

pandas marks missing numbers as `NaN`. Most summaries skip them automatically, so there is no `na.rm = TRUE` to remember. Count them explicitly before you trust any average.

| R | pandas |
| --- | --- |
| `is.na(x)` | `s.isna()` |
| `colSums(is.na(df))` | `df.isna().sum()` |
| `tidyr::drop_na(df, crowd)` | `df.dropna(subset=["crowd"])` |
| `tidyr::replace_na(x, 0)` | `s.fillna(0)` |

In [100]:
matches.isna().sum()

match_id        0
season          0
round           0
date            0
home_team       0
away_team       0
home_goals      0
home_behinds    0
away_goals      0
away_behinds    0
crowd           7
total_goals     0
home_score      0
away_score      0
margin          0
dtype: int64

### Your turn 5.1

Set `n_missing_crowd` to the number of matches with no crowd figure.

In [101]:
n_missing_crowd = matches["crowd"].isna().sum()

In [102]:
check("5.1 n_missing_crowd", lambda: n_missing_crowd, 7)

PASS     5.1 n_missing_crowd


## 6. Sorting and counting

| dplyr | pandas |
| --- | --- |
| `arrange(df, desc(crowd))` | `df.sort_values("crowd", ascending=False)` |
| `count(df, home_team, sort = TRUE)` | `df["home_team"].value_counts()` |
| `slice_max(df, crowd, n = 1)` | `df.nlargest(1, "crowd")` |

In [103]:
matches.sort_values("total_goals", ascending=False)[["round", "home_team", "away_team", "total_goals"]].head()

,round,home_team,away_team,total_goals
151,17,North Melbourne,Collingwood,36
122,14,North Melbourne,Brisbane,35
205,23,Adelaide,Collingwood,35
130,15,Adelaide,Essendon,35
144,17,Geelong,Melbourne,34


In [104]:
matches["home_team"].value_counts().head()

home_team
Essendon      16
Melbourne     15
Gold Coast    15
Richmond      14
GWS           13
Name: count, dtype: int64

### Your turn 6.1

Set `biggest_crowd_team` to the **home team** of the match with the largest crowd, as a plain string.

Hint: `matches["crowd"].idxmax()` returns the row label of the maximum. `matches.loc[row_label, "home_team"]` then reads one cell. (`.loc` gets a proper introduction in Session 7.)

In [105]:
biggest_crowd_team = matches.loc[matches["crowd"].idxmax()]["home_team"]

In [106]:
check("6.1 biggest_crowd_team", lambda: biggest_crowd_team, 'Brisbane')

PASS     6.1 biggest_crowd_team


## Wrap-up

1. Run All and confirm every check prints `PASS`.
2. Commit: `git add sessions/` then `git commit -m "Session 5: first look at pandas"` then `git push`.
3. Spend the rest of the session starting your Milestone 1 notebook from `m1_template.ipynb` (see the Session 5 guide).